# [실행용] 원신 챗봇 실행 (점검 · 시연)

실험 노트북(`01_eval_info` · `02_model_compare` · `03_aux_metrics`)과 다르게, **이 노트북은 완성된 챗봇을 돌리기만 합니다.**
코드는 전부 저장소의 `core/`, `api/`를 그대로 불러 쓰고, 이 노트북에는 설정과 실행 순서만 있습니다.

| 칸 | 하는 일 | 필요할 때 |
|---|---|---|
| 1. 설정 | 저장소 main 받기, 코드·데이터·평가셋 버전 출력 | 항상 |
| 2. 모델 | Qwen2.5-3B 불러오기 (한 번만) | 항상 |
| 3. 점검 | 대표 질문 15개를 돌려 기대한 답이 나오는지 표시, 결과를 드라이브 `results/`에 저장 | 코드·데이터가 바뀐 뒤 |
| 4. 시연 서버 | 같은 모델로 API 서버 + ngrok 주소 열기 (웹 화면 `API_URL`에 넣는 주소) | 시연·발표 |
| 5. 서버 끄기 | 서버와 ngrok 주소 닫기 | 시연 끝 |
| 6. 평가 결과 보기 | 드라이브 `results/`의 공식 평가 결과(info_v2)를 표 하나로 | 발표 자료 |

**준비**
- 런타임 → 런타임 유형 변경 → **T4 GPU**
- 코랩 왼쪽 열쇠 아이콘(보안 비밀): `NGROK_TOKEN`(4번에 필요), `NGROK_DOMAIN`(고정 주소, 선택), `DATABASE_URL`(대화 기록 저장, 선택)
- 토큰·주소 값은 셀에 직접 쓰지 않습니다. 이 노트북을 깃허브에 바로 저장하지 않습니다 (고치면 드라이브에 사본 저장)

In [ ]:
# 1. 설정: 저장소 받기 + 버전 확인
import json, os, subprocess, sys, time
from datetime import datetime
from pathlib import Path

BRANCH = "main"          # PR을 시험할 때만 브랜치 이름으로 바꾼다
REPO = "https://github.com/genshin-build-alchemist/genshin-build-alchemist.git"
REPO_DIR = Path("/content/genshin-build-alchemist")
IN_COLAB = "google.colab" in sys.modules

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

# 예전에 불러온 저장소 모듈 지우기 (안 지우면 셀을 다시 돌려도 옛 코드로 돈다)
for name in [m for m in sys.modules if m.split(".")[0] in ("core", "api", "collector", "db")]:
    del sys.modules[name]
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

from core.kb import load_kb
from core.answer import LLMAnswerer
from core.pipeline import Chatbot

KB = load_kb(REPO_DIR / "data")
evalset = json.loads((REPO_DIR / "data" / "eval" / "questions_info_v2.json").read_text(encoding="utf-8"))
print(f"코드     : {BRANCH} @ {COMMIT}")
print(f"데이터   : {KB.data_version} / 캐릭터 {len(KB.characters)}명, 무기 {len(KB.weapons)}개, 성유물 {len(KB.artifacts)}세트")
print(f"패치     : {KB.version_range} ({len(KB.patches)}건) / 빌드 카드 {len(KB.build_cards)}장")
print(f"평가셋   : {evalset['dataset_version']} {len(evalset['questions'])}문항 (지문 {evalset['sha256'][:12]})")

# 드라이브 (결과 저장용). 팀 폴더가 안 보이면 결과는 현재 폴더에 저장된다
TEAM_DIR = Path("/content/drive/MyDrive/Genshin Build Alchemist (팀)")
if IN_COLAB and not TEAM_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
RESULT_DIR = TEAM_DIR / "results" if TEAM_DIR.exists() else Path("/content/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"결과 저장: {RESULT_DIR}")


def secret(name):
    """코랩 보안 비밀 -> 없으면 환경변수 -> 없으면 None. 값은 출력하지 않는다."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

In [ ]:
# 2. 모델 불러오기 (처음 한 번은 내려받느라 2~3분. 다시 실행하면 이미 불러온 모델을 그대로 쓴다)
from core.llm import load_generate, make_generate

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
if "GEN" not in globals():
    if "model" in globals() and "tokenizer" in globals():   # 다른 셀에서 이미 불러온 모델이 있으면 재사용 (GPU 메모리 절약)
        import torch
        GEN = make_generate(tokenizer, model, torch)
    else:
        GEN = load_generate(MODEL_ID)
ANSWERER = LLMAnswerer(GEN, name=MODEL_ID)            # rag_v4 프롬프트 + 후처리, 패치·아이템은 문장 답
BOT = Chatbot(KB, ANSWERER)
print("답변기:", ANSWERER.name, "/", ANSWERER.method, "/ 후처리", ANSWERER.postprocess)

## 3. 점검

경로마다 대표 질문을 돌립니다. `기대`가 있는 질문은 답(메시지)에 그 말이 들어 있으면 ✓, 없으면 ✗.
✗가 나오면 원문(`raw`)과 근거를 보고 원인을 찾습니다. 결과는 드라이브 `results/run_check_날짜.json`에 저장됩니다.

In [ ]:
# 3. 점검 질문 돌리기
CHECKS = [   # (질문, 기대: 메시지에 들어 있어야 하는 말 / None이면 눈으로 확인)
    ("미탸의 원소는 뭐야?", "번개"),
    ("발레리는 어떤 무기를 써?", "한손검"),
    ("산드로네의 출신 지역은 어디야?", "스네즈나야"),
    ("카에데하라 카즈하의 원소는 뭐야?", "바람"),
    ("4성 번개 원소 캐릭터를 모두 알려줘", "발레리"),
    ("스네즈나야 출신 5성 캐릭터를 모두 알려줘", "미탸"),
    ("아르셀리아의 원소는 뭐야?", "찾지 못했어요"),           # 없는 캐릭터: 지어내지 않기
    ("7.2 신규 캐릭터 누구야?", "미탸, 발레리"),
    ("7.2 신규 무기 뭐야?", "별빛 계시"),
    ("7.2에서 뭐 바뀌었어?", "보댜니차"),
    ("5.0 신규 캐릭터 누구야?", "카치나"),
    ("나히다 패치로 뭐 바뀌었어?", "명의 자리 2"),
    ("7.0에서 산드로네 원소 폭발 바뀌었어?", "원소 폭발"),
    ("별빛 계시 효과 뭐야?", "수성의 궤적"),
    ("카에데하라 카즈하 빌드 추천해줘", None),
]

rows = []
for q, expect in CHECKS:
    start = time.perf_counter()
    r = BOT.ask(q)
    sec = round(time.perf_counter() - start, 2)
    ok = None if expect is None else expect in r["message"]
    rows.append({"question": q, "expect": expect, "ok": ok, "route": r["route"], "message": r["message"],
                 "answer": r["answer"], "evidence": [e["title"] for e in r["evidence"]],
                 "raw": ANSWERER.last_raw, "seconds": sec})
    mark = {True: "✓", False: "✗", None: "·"}[ok]
    print(f"{mark} [{r['route']}, {sec}초] {q}")
    print("   " + r["message"].replace("\n", "\n   "))

checked = [x for x in rows if x["ok"] is not None]
print(f"\n기대한 답: {sum(x['ok'] for x in checked)}/{len(checked)}  "
      f"평균 {sum(x['seconds'] for x in rows) / len(rows):.2f}초, 최대 {max(x['seconds'] for x in rows):.2f}초")

out = RESULT_DIR / f"run_check_{datetime.now():%Y-%m-%d_%H%M}.json"
out.write_text(json.dumps({"code": f"{BRANCH}@{COMMIT}", "data_version": KB.data_version, "model": MODEL_ID,
                           "created_at": datetime.now().isoformat(timespec="seconds"), "rows": rows},
                          ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", out)

## 4. 시연 서버

2번에서 불러온 모델을 그대로 써서 API 서버(`api/main.py`)를 띄우고 ngrok 주소를 엽니다. 모델을 또 불러오지 않아서 T4 메모리가 남습니다.
- 출력된 주소를 웹 화면(Streamlit Cloud)의 Secrets `API_URL`에 넣습니다 (`demo/README.md`). `NGROK_DOMAIN`(고정 주소)을 쓰면 매번 바꾸지 않아도 됩니다
- 주소 + `/docs`로 들어가면 브라우저에서 바로 질문을 보내 볼 수 있습니다
- 코랩 탭을 닫거나 런타임이 끊기면 서버도 꺼집니다. 시연 전에 3번 점검을 한 번 돌려 두면 첫 답이 빨라집니다

In [ ]:
# 4. API 서버 + ngrok 주소 열기
import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "pyngrok"], check=True)

import threading
import urllib.request
import uvicorn
from pyngrok import ngrok

PORT = 8000
if secret("DATABASE_URL"):                       # 대화 기록 저장 (없으면 저장만 건너뛴다)
    os.environ["DATABASE_URL"] = secret("DATABASE_URL")
os.environ["GBA_ANSWERER"] = "mock"              # 불러올 때 모델을 또 올리지 않게 하고, 아래에서 2번 모델로 바꾼다
import api.main as api_main
api_main.bot = BOT

if "SERVER" in globals():                        # 다시 실행하면 이전 서버를 끄고 새로 띄운다
    SERVER.should_exit = True
    time.sleep(2)
SERVER = uvicorn.Server(uvicorn.Config(api_main.app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=SERVER.run, daemon=True).start()
for _ in range(30):
    try:
        health = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2).read())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("서버가 30초 안에 뜨지 않았습니다. 위 오류를 확인하세요.")
print("서버:", health)

token = secret("NGROK_TOKEN")
if not token:
    raise RuntimeError("보안 비밀에 NGROK_TOKEN이 없습니다 (ngrok 대시보드 -> Your Authtoken).")
ngrok.set_auth_token(token)
ngrok.kill()                                     # 이전 주소 정리 (무료 계정은 동시에 주소 1개)
domain = secret("NGROK_DOMAIN")
TUNNEL = ngrok.connect(PORT, domain=domain) if domain else ngrok.connect(PORT)
print("API_URL =", TUNNEL.public_url)
print("질문 화면:", TUNNEL.public_url + "/docs")

In [ ]:
# 5. 서버 끄기 (시연이 끝나면)
from pyngrok import ngrok
ngrok.kill()
if "SERVER" in globals():
    SERVER.should_exit = True
print("서버와 ngrok 주소를 닫았습니다.")

## 6. 평가 결과 보기

공식 평가는 `01_eval_info.ipynb`(채점)에서 돌리고, 결과 JSON은 드라이브 `results/`에 저장됩니다. 여기서는 그 결과를 모아 보기만 합니다.
`DATASET`을 `info_v1`로 바꾸면 7.1 데이터 기준 예전 결과를 봅니다 (두 버전 점수는 직접 비교하지 않습니다).

In [ ]:
# 6. 평가 결과 모아 보기
import pandas as pd

DATASET = "info_v2"
rows6 = []
for path in sorted((TEAM_DIR / "results").glob("*.json")) if TEAM_DIR.exists() else []:
    try:
        d = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if not isinstance(d, dict) or not str(d.get("dataset_version", "")).startswith(DATASET) or "metrics" not in d:
        continue
    m = d["metrics"]
    rows6.append({"파일": path.name, "방법": d.get("method"), "세트": d["dataset_version"],
                  "문항": m.get("n"), "macro F1": m.get("macro_f1"), "빈 답": m.get("empty_answers"),
                  "평균 초": m.get("avg_seconds"), "코드": (d.get("config") or {}).get("code_version")})
if rows6:
    display(pd.DataFrame(rows6).sort_values(["세트", "macro F1"], ascending=[True, False]).reset_index(drop=True))
else:
    print(f"{DATASET} 결과가 아직 없습니다. 01_eval_info.ipynb로 평가를 돌리면 여기에 나옵니다.")